In [11]:
import pandas as pd
import numpy as np

# =====================================
# 1. LOAD DATA
# =====================================
def load_data(file_path, time_col=None):
    df = pd.read_csv(file_path)

    
    if time_col and time_col in df.columns:
        try:
            df[time_col] = pd.to_datetime(df[time_col])
            df = df.sort_values(by=time_col)
        except Exception:
            
            pass

    return df


# =====================================
# 2. EXTRACT MULTI-SENSOR DATA
# =====================================
def get_sensor_data(df, time_col=None):
    """
    Remove time column and keep only numeric sensor data
    """
    if time_col and time_col in df.columns:
        sensor_df = df.drop(columns=[time_col])
    else:
        sensor_df = df.copy()

    sensor_df = sensor_df.select_dtypes(include=[np.number])

    return sensor_df


# =====================================
# 3. CREATE ROLLING WINDOWS
# =====================================
def create_windows(sensor_df, window_size, step_size):
    """
    Each window = one chunk of time-series data
    Supports overlapping using step_size
    """
    windows = []
    n = len(sensor_df)

    for start in range(0, n - window_size + 1, step_size):
        end = start + window_size

        window = sensor_df.iloc[start:end].reset_index(drop=True)
        windows.append(window)

    return windows


# =====================================
# 4. STRUCTURE WINDOWS
# =====================================
def structure_windows(windows):
    """
    Convert to 3D array:
    (num_windows, window_size, num_sensors)
    """
    return np.array([w.values for w in windows])


# =====================================
# 5. CORRELATION STAGE
# =====================================
def compute_correlations(windows):
    """
    Compute correlation matrix for each window
    """
    results = []

    for i, w in enumerate(windows):
        corr = w.corr()
        results.append({
            "window_id": i,
            "correlation_matrix": corr
        })

    return results


# =====================================
# 6. PIPELINE
# =====================================
def rolling_window_pipeline(file_path,
                            window_size=50,
                            step_size=25,
                            time_col=None):

    df = load_data(file_path, time_col)

    sensor_df = get_sensor_data(df, time_col)

    windows = create_windows(sensor_df, window_size, step_size)

    structured_data = structure_windows(windows)

    correlations = compute_correlations(windows)

    return windows, structured_data, correlations


# =====================================
# 7. RUN
# =====================================
if __name__ == "__main__":
    file_path = "complex.csv"   

    windows, structured_data, correlations = rolling_window_pipeline(
        file_path,
        window_size=50,
        step_size=25,
        time_col="time"   
    )

    print("Number of windows:", len(windows))
    print("Structured data shape:", structured_data.shape)

    print("\nFirst window (Sensor Data Only):")
    print(windows[0])

    print("\nCorrelation (Window 0):")
    print(correlations[0]["correlation_matrix"])

Number of windows: 39
Structured data shape: (39, 50, 3)

First window (Sensor Data Only):
          s1        s2        s3
0   1.000000  2.000000  0.700000
1   1.010000  1.999950  0.707000
2   1.019999  1.999800  0.713999
3   1.029996  1.999550  0.720997
4   1.039989  1.999200  0.727993
5   1.049979  1.998750  0.734985
6   1.059964  1.998201  0.741975
7   1.069943  1.997551  0.748960
8   1.079915  1.996802  0.755940
9   1.089879  1.995953  0.762915
10  1.099833  1.995004  0.769883
11  1.109778  1.993956  0.776845
12  1.119712  1.992809  0.783799
13  1.129634  1.991562  0.790744
14  1.139543  1.990216  0.797680
15  1.149438  1.988771  0.804607
16  1.159318  1.987227  0.811523
17  1.169182  1.985585  0.818428
18  1.179030  1.983844  0.825321
19  1.188859  1.982004  0.832201
20  1.198669  1.980067  0.839069
21  1.208460  1.978031  0.845922
22  1.218230  1.975897  0.852761
23  1.227978  1.973666  0.859584
24  1.237703  1.971338  0.866392
25  1.247404  1.968912  0.873183
26  1.257081  1.96